# 📓 Semana 11 · Dia 5 — Primeiro RAG completo com LangChain

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | RAG de produtos respondendo perguntas |

---


## 📖 Teoria — O padrão RAG

**RAG (Retrieval-Augmented Generation)** = recuperar + gerar:

```
pergunta → retrieve (índice vetorial) → contexto
         → generate (LLM + contexto) → resposta com fonte
```

Vantagens: respostas baseadas nos SEUS dados, sem fine-tuning, com citação de fonte. É o padrão #1 em GenAI empresarial.


## 📖 Teoria — LangChain

O **LangChain** é o framework de orquestração de LLMs: chains, retrievers, tools e agents. No Databricks, integra-se com Vector Search e FMA nativamente.


### 💻 Na prática — Montando o RAG

Conecte o retriever (Vector Search) ao LLM (FMA).


In [ ]:
# 1) Retriever via Vector Search
from databricks.vector_search.client import VectorSearchClient
from langchain_community.retrievers import DatabricksVectorSearch
vsc = VectorSearchClient()
retriever = DatabricksVectorSearch(
    vsc.get_index("workspace.prata.produtos_rag_index"),
    columns=["StockCode", "texto"])
print("Retriever criado.")

In [ ]:
# 2) LLM via FMA
from langchain_community.chat_models import ChatDatabricks
llm = ChatDatabricks(endpoint="databricks-llama-3-1-70b", temperature=0.1)
print("LLM conectado.")

In [ ]:
# 3) Chain RAG: recupera contexto e gera resposta
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Responda com base SOMENTE no contexto. Cite o código do produto.\n\nContexto:\n{context}"),
    ("human", "{input}")])
chain_docs = create_stuff_documents_chain(llm, prompt)
rag = create_retrieval_chain(retriever, chain_docs)
print("Chain RAG montada.")

In [ ]:
# 4) Perguntar ao RAG
resp = rag.invoke({"input": "Quais produtos de vidro existem para servir bebidas?"})
print("Resposta:", resp["answer"][:400])
print("Fontes:", [d.metadata.get("StockCode") for d in resp["context"]])

### 💻 Na prática — Analisando o resultado

Observe: a resposta veio do contexto recuperado? As fontes citam códigos reais? Isso é o que a avaliação da Semana 12 vai medir.


> 🎯 **Dica de prova**: GenAI Assoc (App Dev ~25%): LangChain + FMA + Vector Search é o trio. Pergunta: 'qual framework orquestra retrieval+geração?' → LangChain.


## 🎯 Exercícios de fixação

**1.** Faça 5 perguntas ao RAG e anote as respostas.

**2.** Por que o RAG não precisa de fine-tuning para responder sobre os dados?

**3.** O que acontece se o retriever retorna chunks irrelevantes?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Perguntas

Ex.: 'produtos para cozinha', 'item mais caro', etc. Avalie se as respostas usam o contexto.

**2.** Sem fine-tuning

O RAG injeta o contexto no prompt — o modelo não precisa 'decorar' os dados; o conhecimento vem do retrieval.

**3.** Chunks ruins

A resposta fica com ruído ou alucina — por isso a qualidade do retrieval (chunking + rerank) domina a qualidade do RAG.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*